# 00 · Setup

**Purpose**
- Connect Colab to Google Drive
- Clone the GitHub repo into Drive (first run) or update it
- Install packages and check the GPU
- Test the error log and the GitHub push

**Run once per Colab session.** Later notebooks start with the short setup cell in Step 4.

## 1 · Mount Google Drive
- The repo and all data live in Drive, so nothing is lost when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone or update the repo
- Fill in `GITHUB_USER` once.
- The token comes from **Colab secrets** (🔑 icon in the left sidebar), key `GITHUB_TOKEN`.
- The token is used for cloning only and then removed from the remote URL, so it is never saved in Drive.

In [ ]:
import os, subprocess
from google.colab import userdata

GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <- edit once
REPO        = "cw-mpvr"
REPO_DIR    = f"/content/drive/MyDrive/{REPO}"

token = userdata.get("GITHUB_TOKEN")
clean_url = f"https://github.com/{GITHUB_USER}/{REPO}.git"

if not os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clean_url], check=True)
    print("Cloned into", REPO_DIR)
else:
    print("Repo already in Drive:", REPO_DIR)

%cd {REPO_DIR}
!git config user.name  "Dedeepya Korukonda"
!git config user.email "YOUR_GITHUB_NOREPLY_EMAIL"   # GitHub > Settings > Emails
!git log --oneline -3

## 3 · Install packages
- PyTorch is already on Colab; this adds the rest from `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

## 4 · Standard setup cell
- **Copy this cell to the top of every notebook** (after mounting Drive).
- Makes `src/` importable, loads the config, fixes the random seed, reports the GPU.

In [ ]:
import sys
REPO_DIR = "/content/drive/MyDrive/cw-mpvr"
%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%load_ext autoreload
%autoreload 2

from src.utils import load_config, set_seed, gpu_info
from src.errorlog import log_error
from src.gitsync import push, pull

cfg = load_config()
set_seed(cfg["seed"])
print(gpu_info())

## 5 · Check the folders
- `cw-mpvr/` holds code (in git).
- `cw-mpvr-data/` holds GQA files, images and cached model outputs (**not** in git, too large).

In [ ]:
for k, v in cfg["paths"].items():
    print(f"{k:6s} -> {v}  (exists: {os.path.isdir(v)})")

## 6 · How to log an error
- When a cell fails: fix it, then run `log_error(...)` in a new cell.
- The error message is captured automatically from the failed cell.
- Everything goes to `ERROR_LOG.md`, which becomes the "Errors and fixes" section of the report.

Example (commented out so it does not create a fake entry):

In [ ]:
# log_error(
#     where="00_setup / step 3",
#     cause="Package version clash between transformers and accelerate",
#     fix="Pinned accelerate>=0.33 in requirements.txt",
#     lesson="Pin versions once the pipeline works",
# )

## 7 · Save work to GitHub
- Save the notebook first (**Ctrl+S**), then push.
- Use short commit messages that say what changed.

In [ ]:
push("Run setup notebook on Colab")